In [1]:
!pip install transformers==4.39.3 accelerate==0.27.2 peft==0.10.0 bitsandbytes datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 3.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 66.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 22.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import torch
import json

2025-08-03 19:35:15.582558: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754249715.851856      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754249715.930335      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
dataset_path = "/kaggle/input/augmented-dataset/augmented_dataset_v1.json"
with open(dataset_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")
print("Sample:", data[0])

train_data = [
    {
        "input_text": f"Generate 3 domain names for this business. Business: {item['business_description']}\nDomains:",
        "target_text": ", ".join(item['expected_domain_names'])
    }
    for item in data
]

dataset = Dataset.from_list(train_data)
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

Loaded 3000 samples
Sample: {'business_description': 'startup for pets in Singapore', 'expected_domain_names': ['startuppets.biz', 'startuppets.io', 'singaporeio.co']}
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2700
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 300
    })
})


In [4]:
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",           # Auto GPU placement
    torch_dtype=torch.float16,   # Half precision
    load_in_4bit=True            # 4-bit quantization
)
print("Loaded Phi-2 in 4-bit for LoRA fine-tuning")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded Phi-2 in 4-bit for LoRA fine-tuning


In [5]:
def tokenize(batch):
    inputs = tokenizer(batch["input_text"], padding="max_length", truncation=True, max_length=128)
    targets = tokenizer(batch["target_text"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
print(tokenized_dataset)

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2700
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 300
    })
})


In [6]:
# Define hyperparameter combinations for tuning
search_space = [
    {"learning_rate": 1e-4, "r": 8, "alpha": 16, "dropout": 0.05},
    {"learning_rate": 2e-4, "r": 16, "alpha": 32, "dropout": 0.05},
    {"learning_rate": 5e-4, "r": 32, "alpha": 64, "dropout": 0.1},
]

In [7]:
from transformers import TrainingArguments, Trainer
from peft import prepare_model_for_kbit_training

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

for i, cfg in enumerate(search_space):
    print(f"\n🔍 Running experiment {i+1} with {cfg}")

    # Load base Phi-2 model in 4-bit
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", load_in_4bit=True, torch_dtype=torch.float16
    )

    # Prepare for k-bit training
    model = prepare_model_for_kbit_training(model)

    # Apply LoRA config for this experiment
    lora_config = LoraConfig(
        r=cfg["r"],
        lora_alpha=cfg["alpha"],
        lora_dropout=cfg["dropout"],
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"./phi2_lora_exp{i+1}",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=cfg["learning_rate"],
        num_train_epochs=2,  # keep 1 for Colab runtime; adjust if you have time
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="paged_adamw_8bit",
        report_to="none"
    )

    # Trainer setup
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    # Train and save
    trainer.train()
    model.save_pretrained(f"./phi2_lora_exp{i+1}")
    tokenizer.save_pretrained(f"./phi2_lora_exp{i+1}")


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.



🔍 Running experiment 1 with {'learning_rate': 0.0001, 'r': 8, 'alpha': 16, 'dropout': 0.05}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:457: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This

Step,Training Loss
10,4.809800
20,3.324000
30,2.335100
40,1.493100
50,1.276800
60,1.146600
70,1.048500
80,0.945300
90,0.835800
100,0.805900


...


🔍 Running experiment 3 with {'learning_rate': 0.0005, 'r': 32, 'alpha': 64, 'dropout': 0.1}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, 

Step,Training Loss
10,2.723500
20,0.949800
30,0.788300
40,0.649100
50,0.656900
60,0.640900
70,0.601400
80,0.546500
90,0.522400
100,0.492400


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `

In [8]:
!zip -r phi2_lora_models.zip phi2_lora_exp1 phi2_lora_exp2 phi2_lora_exp3

  adding: phi2_lora_exp1/ (stored 0%)
  adding: phi2_lora_exp1/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 72%)
  adding: phi2_lora_exp1/README.md (deflated 66%)
  adding: phi2_lora_exp1/adapter_config.json (deflated 51%)
  adding: phi2_lora_exp1/added_tokens.json (deflated 84%)
  adding: phi2_lora_exp1/vocab.json (deflated 59%)
  adding: phi2_lora_exp1/checkpoint-674/ (stored 0%)
  adding: phi2_lora_exp1/checkpoint-674/tokenizer.json (deflated 72%)
  adding: phi2_lora_exp1/checkpoint-674/README.md (deflated 66%)
  adding: phi2_lora_exp1/checkpoint-674/scheduler.pt (deflated 55%)
  adding: phi2_lora_exp1/checkpoint-674/adapter_config.json (deflated 51%)
  adding: phi2_lora_exp1/checkpoint-674/added_tokens.json (deflated 84%)
  adding: phi2_lora_exp1/checkpoint-674/vocab.json (deflated 59%)
  adding: phi2_lora_exp1/checkpoint-674/rng_state.pth (deflated 25%)
  adding: phi2_lora_exp1/checkpoint-674/trainer_state.json (deflated 79%)
  adding: phi2_lora_exp1/checkpoint-674/tokenizer_config.json (deflated 94%)
  adding: phi2_lora_exp1/checkpoint-674/training_args.bin (deflated 51%)
  

In [9]:
from IPython.display import FileLink

# Create a link to download the zip file
FileLink(r'phi2_lora_models.zip')

/kaggle/working/phi2_lora_models.zip